# The prompt workbench

One afternoon at a time, with the prompt open beside it. What to change is decided here; whether it worked is answered by `python -m research.run`.

Three things to know before reading a result:

1. **One afternoon is n=1.** On 3 September a set of changes that read as improvements moved all eight axes down, 3.49 → 2.93.
2. **The loop is not fast.** A devise call measures 76–184 s. What this buys is the state between calls and a look at each step.
3. **Editing a block here edits the repository.** When one is settled: commit, `python -m tools.prompts --write`, then a run.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import research.bench as bench  # noqa: E402 - the root has to be on the path first

print(sorted(bench.environment()))
print("prompt fingerprint:", bench.reload_prompts())

# One row per call, so a comparison is not a memory of how the last one read.
TRIED: list[dict[str, object]] = []

Credentials are not in `env.ps1`: the router builds a `DefaultAzureCredential`, which finds the `az` session in the `AZURE_CONFIG_DIR` named there. If the first model call fails on authentication, run `az account get-access-token` in a terminal with that variable set — `az account show` reads the cache and succeeds even on an expired token.

## 1. The blocks the prompt is made of

In [ ]:
for module in (
    "agents.experience_deviser",
    "shared.experience_prompt",
    "agents.experience_continuer",
):
    said = bench.blocks(module)
    print(f"{module}  —  {len(said)} blocks, {sum(n for _, n in said)} characters")
    for name, size in said:
        print(f"    {name:22} {size:6}")
    print()

In [ ]:
BLOCK = ("agents.experience_deviser", "manner-tail")
print(bench.read(*BLOCK))

## 2. The household

Everything that decides one afternoon, as one dictionary. `research/run.py` sends it to `devise_experience`; the next cell gives the same one to `the_prompt`.

In [ ]:
from research.households import Household, Memory, arguments

house = Household(
    name="bench",
    interests=("i treni", "le mappe vecchie"),
    avoid=("i ragni",),
    load="middle",
    ink="middle",
    span="middle",
    sheets=2,
    note="",
)
memory = Memory()  # no history: the first afternoon of this house
args = arguments(house, memory)

for name, value in args.items():
    shown = value if isinstance(value, str) else repr(value)
    print(f"{name:12} {shown[:110] or '—'}")

## 3. The whole prompt, before paying for it

A `form` and a `move` out of `methods/`, filtered to what this house can run. Drawn here; in the real call the model chooses them and a draw is the fallback. Which pair was used is in `built_from` below.

In [ ]:
from agents.experience_deviser import the_prompt
from shared.methods import draw, load, runnable

here = runnable(load(), capabilities=args["capabilities"])
form, move = draw(here)
print(f"{len(here)} methods this house can run · {form.method_id} + {move.method_id}\n")

prompt = the_prompt(**args, form=form, move=move)
print(f"{len(prompt)} characters in all\n")
print(prompt[-3000:])

## 4. The call

The whole path: choosing the method, the call, the format, one repair if it is needed, the checks, the safety gate. Logging is on for the repairs — a check that fires every time is a defect in the prompt, and nothing else shows it.

Measured: 76–184 s, median about 140.

In [ ]:
import logging
import time

from panel.devising import RefusedByTheChecks, devise_experience
from shared.errors import SafetyBlocked
from shared.experience import ExperienceError

logging.basicConfig(level=logging.WARNING, force=True)
logging.getLogger("panel.devising").setLevel(logging.INFO)

built: dict[str, str] = {}
began = time.time()
row: dict[str, object] = {"prompt": bench.reload_prompts(), "household": house.name}
try:
    experience, spent = await devise_experience(**args, built_from=built, now=began)
    document = experience.to_dict()
    row |= {"title": experience.title, "moments": len(experience.moments)}
except (RefusedByTheChecks, ExperienceError, SafetyBlocked) as exc:
    experience = document = None
    row["refused"] = f"{type(exc).__name__}: {exc}"
row |= {"seconds": round(time.time() - began, 1), **built}
TRIED.append(row)
row

In [ ]:
from tools.as_it_arrives import read

counted = read(document)

## 5. The checks

Almost always empty: `devise_experience` repairs once before it returns. The complaints there were are in the log above, and a complaint that comes back every afternoon names the rule to work on.

In [ ]:
from shared.experience_checks import check

complaints = check(experience, recent=args["recent"], sheets_at_most=args["sheets"])
print("; ".join(map(str, complaints)) or "no complaints")

## 6. Played with nobody in the room

`research/play.py` walks the moments the way the house does and asks a model, standing in for the person, what came back on the sheet. No page is drawn — a sheet arrives as the words that would be printed on it — and an `ask` branch ends the run instead of buying a continuation.

The mood is the one dial, and it is a property of a day: it is what reaches the branch where a sheet comes back blank.

In [ ]:
from research.calls import a_context
from research.play import play
from shared.experience import Weight

ctx = a_context(time.time())
played = await play(
    ctx,
    experience=experience,
    household=house.name,
    weight=Weight.STANDARD,
    mood="una giornata normale, c'è voglia di fare qualcosa",
)
print(f"ended: {played.ending} · {played.minutes} min · stopped at {played.reached}\n")
print(played.transcript())

## 7. The eight axes, on this one afternoon

Every axis has to quote a line word for word, and that is the part a prompt can be changed against.

In [ ]:
from research.calls import appraise

scored = await appraise(ctx, transcript=played.transcript())
for axis, said in (scored.get("axes") or {}).items():
    print(f"{said.get('score')}  {axis:26} {said.get('says', '')}")
print("\nto the prompt:", scored.get("whatToChangeInThePrompt", "—"))

## 8. The loop

Edit the `.md`, save, run the cell below, go back to step 4. The fingerprint changes only when the text being sent changed: two afternoons under the same one were written by the same prompt, and two under different ones do not compare however alike they read.

`bench.write` puts a block back from here when that is easier; the cell reads from disk either way.

In [ ]:
print("fingerprint:", bench.reload_prompts())

for one in TRIED:
    print(one)

## 9. When a block is settled

```
python -m tools.prompts --write
python -m pytest -q
python -m research.run --iterations 4 --seed 0 --label <name>
```

The last one is the measurement: six households × four iterations, about an hour and ~0.36 € of tokens, and it writes the prompt fingerprint into the summary.

⚠️ Do not run `pytest` and a run at the same time: `tests/test_trail.py` fails with `Event loop is closed` on contention, and it looks like a real regression.

⚠️ The hub is synchronised by hand. A constraint changed in `shared/` holds in the cloud and not in the house until it is copied: `powershell -NoProfile -File scripts\hub-stale.ps1`.